# Module 3: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the shared order agent (`agents/order_agent.py`), then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, score the agent end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
4. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from utils.models import model
from utils.workshop import scoped, workshop_user
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
import json
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))
print("Workshop user:", workshop_user())


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** an order-triage prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = scoped("order-triage-assistant")
prompt = ChatPromptTemplate([
    ("system",
     "You are an order triage assistant for a medical device order processing team. "
     "Look up the order, then produce a concise triage briefing: "
     "(1) a one-line summary, (2) what is blocking the order, "
     "(3) three concrete next actions, and (4) two questions for the account. "
     "Be factual and neutral; do not give reimbursement or clinical advice."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for `create_agent`, running on the workshop's shared `model`. A small mock `lookup_order` tool lets the agent fetch the order, so the full agent loop (model → tool → model) shows up in the trace.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a medical device order by its order number."""
    # Mock order-system lookup — swap for your real order source in production.
    directory = {
        "PO-4471": (
            "PO-4471, Mercy Regional Medical Center. 2x ambulatory infusion pump, "
            "12x infusion set (6mm, box of 10). Payer: Blue Cross Blue Shield IL. "
            "Insurance code: missing. Prior authorization: pending. Status: on hold. "
            "Account requires lot numbers on every line."
        ),
    }
    return directory.get(order_id, f"No order found for {order_id!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

agent = create_agent(model=model, tools=[lookup_order], system_prompt=system_prompt)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Triage order PO-4471 for me."}]}
)
print(result["messages"][-1].text)

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are an order triage assistant for a medical device order processing team. "
     "Look up the order, then produce a concise triage briefing: "
     "(1) a one-line summary, (2) what is blocking the order, "
     "(3) three concrete next actions, (4) two questions for the account, "
     "and (5) one compliance flag to keep in mind. "
     "Be factual and neutral; do not give reimbursement or clinical advice."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt(f"{prompt_name}:<commit-hash>"),
# and tear down with client.delete_prompt(prompt_name) when you're done.

## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the shared order agent (`agents/order_agent.py`) three times with **deliberately light** prompts — 
each one says "search at most once" so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [ ]:
from agents.order_agent import build_order_agent

agent = build_order_agent()

warmup_prompts = [
    "In one sentence, what is a prior authorization? Search at most once.",
    "In one sentence, what is an HCPCS code used for? Search at most once.",
    "In one sentence, what does DME stand for in medical billing? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [ ]:
project_name = os.environ.get("LANGSMITH_PROJECT", scoped("modular-workshops"))
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", scoped("modular-workshops")),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", scoped("modular-workshops")),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. 
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

### 3.1 Dataset

An eval dataset is just labeled `inputs` + expected `outputs`. Where those rows
*live* is a separate question from where you *run* the experiment. Many teams
already keep their eval data in **Databricks** (Unity Catalog / Delta tables) and
want to keep it there as the source of truth — that's fine. LangSmith doesn't
have to own the data to score against it.

Two ways to work:

- **Local source (default below):** define examples inline for the workshop.
- **Databricks source (§3.1a):** pull the rows from a Delta table over a SQL
  warehouse and map them into the same example shape. Your governance,
  versioning, and lineage stay in Databricks; LangSmith becomes the *evaluation
  and observability* layer on top.

Either way we end up with the same `examples` list — same input set, two
reference shapes (one for final-response judging, one for trajectory matching) —
and load it into a LangSmith dataset the same way.

In [ ]:
# Local (default) source. The Databricks cells in 3.1a can override this
# `examples` list with rows pulled from your Delta table.
examples = [
    {
        "inputs": {"query": "Write a one-line status note for order PO-4471 to /order_note.txt"},
        "outputs": {
            "reference_answer": "A one-line status note for PO-4471 should be saved to /order_note.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research prior-authorization requirements for ambulatory infusion pumps and write a brief exception note to /exception_note.md"},
        "outputs": {
            "reference_answer": "A short exception note on infusion pump prior-authorization requirements, written to /exception_note.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short hold summary for PO-4471 to /hold_summary.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short hold summary for PO-4471 written to /hold_summary.md and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan out what documentation an infusion pump prior authorization needs, then research it and write the findings to /report.md"},
        "outputs": {
            "reference_answer": "A planned-out summary of infusion pump prior-authorization documentation, written to /report.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

### 3.1a (Optional) Databricks as the source of truth

If your eval data already lives in Databricks, keep it there and pull from it.
The flow is: **read creds → (optionally) seed a demo table → query the Delta
table → map rows to the LangSmith example shape**. The result reassigns
`examples`, so the dataset-creation cell below is unchanged.

**Interoperability, not migration.** Databricks stays the system of record for
the rows (governed by Unity Catalog, versioned as a Delta table). LangSmith is
where you run experiments, judge outputs, and observe traces. You can re-pull on
every run so the LangSmith dataset is always a snapshot of the current Delta
table.

**Setup.** Add these to your `.env` (a Databricks **Free Edition** workspace
works). Leave them unset to skip this section and use the local examples above.

```bash
DATABRICKS_HOST="dbc-xxxxxxxx-xxxx.cloud.databricks.com"   # no https://
DATABRICKS_HTTP_PATH="/sql/1.0/warehouses/xxxxxxxxxxxxxxxx" # SQL warehouse
DATABRICKS_TOKEN="dapi..."                                  # personal access token
# Optional — where the demo table lives (defaults shown):
DATABRICKS_CATALOG="workspace"
DATABRICKS_SCHEMA="default"
DATABRICKS_EVAL_TABLE="order_agent_evals"
```

Install the connector once (already fine to run in the workshop env):

```bash
uv pip install databricks-sql-connector
```

In [ ]:
# --- Databricks config (read from .env; graceful skip if unset) ---
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_HTTP_PATH = os.environ.get("DATABRICKS_HTTP_PATH")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")

DBX_CATALOG = os.environ.get("DATABRICKS_CATALOG", "workspace")
DBX_SCHEMA = os.environ.get("DATABRICKS_SCHEMA", "default")
DBX_TABLE = os.environ.get("DATABRICKS_EVAL_TABLE", "order_agent_evals")
DBX_FQN = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"

databricks_configured = all([DATABRICKS_HOST, DATABRICKS_HTTP_PATH, DATABRICKS_TOKEN])


def databricks_connection():
    """Open a Databricks SQL warehouse connection. Requires databricks-sql-connector."""
    from databricks import sql  # pip install databricks-sql-connector

    return sql.connect(
        server_hostname=DATABRICKS_HOST,
        http_path=DATABRICKS_HTTP_PATH,
        access_token=DATABRICKS_TOKEN,
    )


if databricks_configured:
    print("Databricks configured. Source table:", DBX_FQN)
else:
    print("Databricks not configured (DATABRICKS_* unset) \u2014 skipping 3.1a.")
    print("The dataset will be built from the local `examples` above.")

### 3.1b (Optional) Seed a demo eval table in your Databricks workspace
Skip this if your team already maintains the source-of-truth table. Schema is chosen to map straight into LangSmith examples: one row per eval case, with the trajectory stored as a JSON string (portable across engines).

In [ ]:
if databricks_configured:
    seed_rows = [
        (e["inputs"]["query"], e["outputs"]["reference_answer"],
         json.dumps(e["outputs"]["trajectory"]))
        for e in examples  # reuse the local examples as seed data
    ]
    with databricks_connection() as conn, conn.cursor() as cur:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {DBX_CATALOG}.{DBX_SCHEMA}")
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {DBX_FQN} (
                query STRING,
                reference_answer STRING,
                trajectory STRING  -- JSON array of tool names
            )
        """)
        # Idempotent reseed: clear then insert the demo rows.
        cur.execute(f"DELETE FROM {DBX_FQN}")
        cur.executemany(
            f"INSERT INTO {DBX_FQN} (query, reference_answer, trajectory) VALUES (?, ?, ?)",
            seed_rows,
        )
    print(f"Seeded {len(seed_rows)} rows into {DBX_FQN}")
else:
    print("Skipped seeding \u2014 Databricks not configured.")

### 3.1c (Optional) Pull from Databricks and map into the LangSmith example shape
Reassigns `examples` so the dataset-creation cell below is identical whether the data came from Databricks or the local default.

In [ ]:
# --- Pull from Databricks and map into the LangSmith example shape ---
# Reassigns `examples` so the dataset-creation cell below is identical whether
# the data came from Databricks or the local default.
def load_examples_from_databricks() -> list[dict]:
    with databricks_connection() as conn, conn.cursor() as cur:
        cur.execute(f"SELECT query, reference_answer, trajectory FROM {DBX_FQN}")
        rows = cur.fetchall()

    mapped = []
    for query, reference_answer, trajectory in rows:
        # `trajectory` is a JSON array string in the table; parse to a list.
        traj = json.loads(trajectory) if isinstance(trajectory, str) else list(trajectory or [])
        mapped.append({
            "inputs": {"query": query},
            "outputs": {"reference_answer": reference_answer, "trajectory": traj},
        })
    return mapped


if databricks_configured:
    examples = load_examples_from_databricks()
    print(f"Pulled {len(examples)} examples from {DBX_FQN}")
    for e in examples[:2]:
        print(" -", e["inputs"]["query"][:70], "->", e["outputs"]["trajectory"])
else:
    print(f"Using {len(examples)} local examples (Databricks not configured).")

In [ ]:
dataset_name = scoped("order-agent-evals")

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")

### 3.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Run the agent and return its response plus any files it wrote.

    The evaluator receives these as separate fields so it can inspect both
    the assistant's message and the resulting file artifacts.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    return {
        "response": result["messages"][-1].text,
        "files": result.get("files") or {},
    }

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant.

You will see:
1. The user's request
2. The assistant's final response
3. Any files written by the assistant
4. A success rubric

Evaluate both the response and the files. If the task asks the assistant to write a file, inspect the file contents when available. Mark `score=True` if the task was completed successfully, even if the wording differs from the rubric.

Mark `score=False` only if the task was clearly missed, the file is missing or incorrect, the response contains errors, or the assistant refused without good reason.

Give one short sentence of reasoning.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Files written: {outputs.get('files', {})}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])

    return {
        "key": "correctness",
        "score": int(grade["score"]),
        "comment": grade["reasoning"],
    }

In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [ ]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

In [ ]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an assistant response satisfied the user's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", scoped("modular-workshops")),
    display_name=scoped("workshop-online-correctness"),
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name=scoped("order-agent-needs-review"),
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


In [ ]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", scoped("modular-workshops")),
    display_name=scoped("workshop-route-correctness"),
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "In one sentence, what is a prior authorization denial? Search at most once.",
    "In one sentence, what is a certificate of medical necessity? Search at most once.",
    "In one sentence, what is the difference between HCPCS and CPT codes? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "tavily_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the shared order agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **3.1a Databricks interop** | Keep eval data in a Delta table; pull + map into LangSmith | `databricks-sql-connector` → same `examples` shape |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

**Going further:** LangSmith **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.